In [1]:
import torch
from PIL import Image
import open_clip

/Users/zhiweizhang/Projects/nazi_symbols_classification/venv/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [10]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
model.eval()  # model in train mode by default, impacts some models with BatchNorm or stochastic depth active

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-11): 12 x ResidualAttentionBlock(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=768, out_features=3072, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=3072, out_features=768, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((768,), eps=1e-05, elementwise_affine

In [2]:
from nazi_symbols_classification.training.data_preparation import get_image_paths

In [3]:
images = get_image_paths("/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-classification", 
                         ("train", "test", "valid"))

In [39]:
train_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-classification/train') and "non-nazi" not in image]
test_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-classification/test') and "non-nazi" not in image]
valid_images = [image for image in images if image.startswith('/Users/zhiweizhang/Projects/nazi_symbols_classification/notebooks/datasets/nazi-symbols-classification/valid') and "non-nazi" not in image]

In [40]:
import os

y_train = [os.path.basename(os.path.dirname(image)) for image in train_images]
y_test = [os.path.basename(os.path.dirname(image)) for image in test_images]
y_valid = [os.path.basename(os.path.dirname(image)) for image in valid_images]

In [41]:
import numpy as np
import pandas as pd


def load_image(image_path):
    with torch.no_grad(), torch.cpu.amp.autocast():
        image = preprocess(Image.open(image_path)).unsqueeze(0)
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        return pd.DataFrame(image_features.float().numpy())

def preprocess_images(images):
    return pd.concat([load_image(image_path) for image_path in images], axis=0)

In [44]:
with open("training_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(train_images), 200):
    data = preprocess_images(train_images[i:i+200])
    with open("training_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [45]:
import json

with open("training_data_multiclass_label.csv", "w") as f:
    json.dump(y_train, f)

In [46]:
with open("validation_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(valid_images), 200):
    data = preprocess_images(valid_images[i:i+200])
    with open("validation_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [47]:
with open("validation_data_multiclass_label.csv", "w") as f:
    json.dump(y_valid, f)

In [48]:
with open("test_data_multiclass.csv", "w") as f:
    f.write(",".join(map(str, range(512)))+"\n")

for i in range(0, len(test_images), 200):
    data = preprocess_images(test_images[i:i+200])
    with open("test_data_multiclass.csv", "a+") as f:
        f.write("\n".join(data.to_csv(None, index=False).split("\n")[1:]))

In [49]:
with open("test_data_multiclass_label.csv", "w") as f:
    json.dump(y_test, f)

In [50]:
training_data = pd.read_csv("training_data_multiclass.csv")
validation_data = pd.read_csv("validation_data_multiclass.csv")
test_data = pd.read_csv("test_data_multiclass.csv")

In [51]:
training_data.head()

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
0,-0.011841,-0.135742,-0.064941,0.030640,0.027222,-0.020264,-0.006287,-0.022095,0.025024,0.098633,...,-0.022949,0.107422,0.013062,-0.015320,0.057373,-0.016602,0.024048,-0.019531,-0.000877,-0.031250
1,0.018921,-0.094238,-0.124023,0.008362,-0.012451,-0.055420,-0.022095,-0.010803,0.020630,0.015503,...,0.022827,0.101074,0.037598,0.027466,0.059570,0.092285,-0.033447,0.044678,0.041992,-0.054932
2,0.045898,-0.084961,-0.088867,-0.000725,-0.026489,-0.011414,0.022339,-0.054932,0.021851,-0.003036,...,0.029663,0.128906,0.007599,0.013306,0.054443,-0.048340,-0.010620,-0.013550,-0.020752,0.009155
3,0.035156,-0.090332,-0.064941,0.009338,-0.000675,-0.040283,0.012878,0.015320,0.015137,-0.031494,...,0.037842,0.103516,0.003418,0.049561,0.033936,0.014832,-0.032715,-0.034912,0.062500,0.001740
4,0.009033,-0.255859,-0.078613,-0.011353,0.004761,-0.025391,0.035156,-0.021606,-0.035400,-0.020508,...,0.016357,0.012390,0.081055,0.033447,-0.002670,0.017822,-0.034180,-0.023804,-0.041016,0.002777


In [52]:
%%time

# import the library
from sklearn.linear_model import LogisticRegression

# instantiate & fit
lr=LogisticRegression(max_iter=5000)
lr.fit(training_data, y_train)

print("score on test: " + str(lr.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.7681660899653979
CPU times: user 1.18 s, sys: 474 ms, total: 1.65 s
Wall time: 91.1 ms


In [53]:
%%time

# import the library
from sklearn.linear_model import SGDClassifier

# instantiate & fit
sgd=SGDClassifier()
sgd.fit(training_data, y_train)

print("score on test: " + str(sgd.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.8096885813148789
CPU times: user 913 ms, sys: 1.28 s, total: 2.19 s
Wall time: 116 ms


In [54]:
y_predict = sgd.predict(pd.concat([validation_data, test_data]))
y_true = y_valid + y_test

In [55]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_predict)
print(report)

                          precision    recall  f1-score   support

               black_sun       0.88      0.71      0.79        62
british_union_of_fascist       1.00      0.67      0.80         6
        broken_sun_cross       1.00      0.10      0.18        10
          happy_merchant       0.86      1.00      0.92        18
                  hitler       0.87      0.81      0.84        97
           hitler_salute       0.00      0.00      0.00         2
              judenstern       1.00      1.00      1.00         1
                neo_nazi       0.59      0.77      0.67        65
                siegrune       0.80      0.59      0.68        79
                ss_skull       0.73      0.82      0.77        99
   sturmabteilung_emblem       1.00      0.75      0.86         4
                swastika       0.86      0.91      0.88       404
              wolfsangel       0.57      0.40      0.47        20

                accuracy                           0.81       867
        

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.


In [56]:
%%time

# import the library
from sklearn.neighbors import KNeighborsClassifier

# instantiate & fit
knn = KNeighborsClassifier(algorithm = 'brute', n_jobs=-1)
knn.fit(training_data, y_train)

print("score on test: " + str(knn.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.7520184544405998
CPU times: user 494 ms, sys: 62.3 ms, total: 556 ms
Wall time: 42.6 ms


In [57]:
%%time

# import the library
from sklearn.svm import LinearSVC

# instantiate & fit
svm=LinearSVC(C=0.0001)
svm.fit(training_data, y_train)

print("score on test: " + str(svm.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.4659746251441753
CPU times: user 110 ms, sys: 8.52 ms, total: 118 ms
Wall time: 83.8 ms


In [58]:
%%time

# import the library
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
clf = DecisionTreeClassifier(min_samples_split=10,max_depth=3)
clf.fit(training_data, y_train)

print("score on test: " + str(clf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5778546712802768
CPU times: user 129 ms, sys: 4.04 ms, total: 133 ms
Wall time: 131 ms


In [59]:
%%time

# import the library
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
bg=BaggingClassifier(DecisionTreeClassifier(min_samples_split=10,max_depth=3),max_samples=0.5,max_features=1.0,n_estimators=10)
bg.fit(training_data, y_train)

print("score on test: " + str(bg.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5870818915801614
CPU times: user 562 ms, sys: 11.2 ms, total: 573 ms
Wall time: 572 ms


In [60]:
%%time

# import the library
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# instantiate & fit
adb = AdaBoostClassifier(DecisionTreeClassifier(max_depth=2),n_estimators=100,learning_rate=0.5)
adb.fit(training_data, y_train)

print("score on test: " + str(adb.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


score on test: 0.4925028835063437
CPU times: user 8.44 s, sys: 134 ms, total: 8.58 s
Wall time: 8.58 s


In [61]:
%%time

# import the library
from sklearn.ensemble import GradientBoostingClassifier

# instantiate & fit
gbc = GradientBoostingClassifier(n_estimators=100)
gbc.fit(training_data, y_train)

print("score on test: " + str(gbc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.7012687427912342
CPU times: user 2min 27s, sys: 224 ms, total: 2min 28s
Wall time: 2min 28s


In [62]:
%%time

# import the library
from sklearn.ensemble import RandomForestClassifier

# instantiate & fit
rf = RandomForestClassifier(n_estimators=300,max_depth=3)
rf.fit(training_data, y_train)

print("score on test: " + str(rf.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.5847750865051903
CPU times: user 1.33 s, sys: 4.55 ms, total: 1.34 s
Wall time: 1.34 s


In [63]:
%%time

# import the library
from sklearn.ensemble import VotingClassifier

# 2) logistic regression =lr
lr=LogisticRegression(max_iter=5000)
# 3) random forest =rf
rf = RandomForestClassifier(n_estimators=30,max_depth=3)
# 4) suport vecotr mnachine = svm
svm=LinearSVC(max_iter=5000)
evc=VotingClassifier(estimators=[('lr',lr),('rf',rf),('svm',svm)])
evc.fit(training_data, y_train)

print("score on test: " + str(evc.score(pd.concat([validation_data, test_data]), y_valid + y_test)))

score on test: 0.7704728950403691
CPU times: user 3.39 s, sys: 1.23 s, total: 4.61 s
Wall time: 599 ms


In [147]:
len(training_images), len(valid_images) + len(test_images)

(6561, 9914)